# Phase 9 dSVA + I-JEPA Validation

Run this notebook on a Colab GPU runtime. It validates the current positive signal by continuing the released dSVA checkpoint with the matched DINO+MAE control and the low-weight I-JEPA variant, then evaluating on the full Imagenette validation split across repeated training seeds.

Default comparison:

- Control: DINO+MAE continuation, `lr=5e-5`
- JEPA: DINO+MAE+I-JEPA continuation, `jepa_weight=0.05`, `lr=5e-5`
- Loss weights normalized with `--normalize-loss-weights`
- Validation limit set to `5000`, which covers the full Imagenette validation split


In [ ]:
import torch

print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


If CUDA is not available, switch Colab to `Runtime > Change runtime type > T4 GPU` or better before continuing.

In [ ]:
%cd /content
!rm -rf jepa-transfer-attacks imagenette2-320 imagenette2-320.tgz
!git clone https://github.com/Tariolle/jepa-transfer-attacks.git
%cd /content/jepa-transfer-attacks
!pip install -q -r requirements.txt


In [ ]:
%cd /content
!wget -q --show-progress https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
!tar -xzf imagenette2-320.tgz

from huggingface_hub import hf_hub_download

DSVA_CHECKPOINT = hf_hub_download(
    repo_id='NexusBohanLiu/dSVA',
    filename='model.pth',
    local_dir='/content/dsva_checkpoint',
)
print('dSVA checkpoint:', DSVA_CHECKPOINT)


Configure the validation. Use `SEEDS = [0]` for a quick first run, then restore repeated seeds for the actual Phase 9 check.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/jepa-transfer-attacks')
TRAIN_ROOT = '/content/imagenette2-320/train'
VAL_ROOT = '/content/imagenette2-320/val'

SEEDS = [0, 1, 2]
TRAIN_LIMIT = 1000
EVAL_LIMIT = 5000
CONFIG = '0.05:0.00005'
OUTPUT_MODE = 'scaled-delta'
OUTPUT_DIR = REPO_ROOT / 'results' / 'phase9_dsva_official_jepa_validation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUTPUT_DIR)


Smoke test the released checkpoint loader and classifier evaluation before launching training continuations.

In [ ]:
import subprocess
import sys

smoke_cmd = [
    sys.executable,
    'scripts/run_dsva_checkpoint_attack.py',
    '--data-root', VAL_ROOT,
    '--checkpoint', DSVA_CHECKPOINT,
    '--output-mode', 'adv',
    '--limit', '16',
    '--batch-size', '8',
    '--epsilon', '0.06274509803921569',
    '--victims', 'resnet50', 'convnext_tiny', 'vit_b_16',
    '--device', 'cuda',
    '--output-csv', 'results/phase9_smoke_released_dsva.csv',
]
subprocess.run(smoke_cmd, cwd=REPO_ROOT, check=True)


Run the matched continuation/evaluation jobs. Each seed produces one control checkpoint and one JEPA checkpoint, plus eval CSVs and a per-seed summary.

In [ ]:
import subprocess
import sys

for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f'seed_{seed}'
    cmd = [
        sys.executable,
        'scripts/sweep_dsva_jepa_finetune.py',
        '--train-root', TRAIN_ROOT,
        '--val-root', VAL_ROOT,
        '--init-checkpoint', DSVA_CHECKPOINT,
        '--output-dir', str(seed_dir),
        '--run-prefix', f'phase9_seed{seed}',
        '--configs', CONFIG,
        '--limit', str(TRAIN_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--batch-size', '1',
        '--eval-batch-size', '8',
        '--grad-accum-steps', '8',
        '--epochs', '1',
        '--epsilon', '0.06274509803921569',
        '--output-mode', OUTPUT_MODE,
        '--victims', 'resnet50', 'convnext_tiny', 'vit_b_16',
        '--device', 'cuda',
        '--num-workers', '0',
        '--seed', str(seed),
        '--normalize-loss-weights',
        '--skip-existing',
    ]
    print('\n' + ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


Aggregate matched gains across seeds.

In [ ]:
import csv
from statistics import mean, stdev

rows = []
for seed in SEEDS:
    summary_path = OUTPUT_DIR / f'seed_{seed}' / f'phase9_seed{seed}_summary.csv'
    with summary_path.open(newline='', encoding='utf-8') as handle:
        seed_rows = list(csv.DictReader(handle))

    control = next(row for row in seed_rows if row['run_type'] == 'control' and row['lr'] == '5e-05')
    jepa = next(row for row in seed_rows if row['run_type'] == 'jepa' and row['jepa_weight'] == '0.05' and row['lr'] == '5e-05')
    control_mean = float(control['mean_transfer_success'])
    jepa_mean = float(jepa['mean_transfer_success'])
    rows.append(
        {
            'seed': seed,
            'control_mean_transfer_success': control_mean,
            'jepa_mean_transfer_success': jepa_mean,
            'matched_gain': jepa_mean - control_mean,
            'control_eval_csv': control['eval_csv'],
            'jepa_eval_csv': jepa['eval_csv'],
        }
    )

aggregate_path = OUTPUT_DIR / 'phase9_matched_gain_summary.csv'
with aggregate_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

gains = [row['matched_gain'] for row in rows]
print(f'Saved aggregate summary: {aggregate_path}')
print(f'Mean matched gain: {mean(gains):.6f}')
if len(gains) > 1:
    print(f'Std matched gain: {stdev(gains):.6f}')

for row in rows:
    print(
        f"seed={row['seed']} control={row['control_mean_transfer_success']:.6f} "
        f"jepa={row['jepa_mean_transfer_success']:.6f} gain={row['matched_gain']:.6f}"
    )


Package CSV and JSON summaries so they can be downloaded from Colab without bundling large checkpoints.

In [ ]:
%cd /content/jepa-transfer-attacks
!find results/phase9_dsva_official_jepa_validation -type f \( -name '*.csv' -o -name '*.json' \) -print > results/phase9_files_to_archive.txt
!printf '%s\n' results/phase9_smoke_released_dsva.csv >> results/phase9_files_to_archive.txt
!zip results/phase9_dsva_official_jepa_validation_summaries.zip -@ < results/phase9_files_to_archive.txt
!ls -lh results/phase9_dsva_official_jepa_validation_summaries.zip
